┌─────────────────────────────────────────────────────────┐
│                    DATA LAYER                           │
│  (Your existing pipeline — historical + live streaming) │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  FEATURE LAYER                          │
│  (Alignment, resampling, all engineered features)       │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  SIGNAL LAYER                           │
│  (Individual setups A/B/C/D — each independently)      │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│                  RISK LAYER                             │
│  (Position sizing, max drawdown, correlation limits)    │
└────────────────────────┬────────────────────────────────┘
                         │
┌────────────────────────▼────────────────────────────────┐
│               EXECUTION LAYER                           │
│  (Order routing, slippage, Binance API)                 │
└─────────────────────────────────────────────────────────┘

In [ ]:
from orderflow import futures_agg_trades, get_oi, get_mark_price_klines, spot_agg_trades, get_premium_index_klines
import pandas as pd
from datetime import datetime, timedelta

symbol = "BTCUSDT"
date = "2026-06-10"

fut_trades = futures_agg_trades(symbol, date)
oi = get_oi(symbol, date)
context = get_mark_price_klines(symbol, date)
spot = spot_agg_trades(symbol, date)

mark_5m = context.resample('5min').agg({"open":'first',"high":"max", "low":"min", "close":"last"})


import numpy as np
I = 0.0001  # 0.01% per 8h — see caveat below, this is BTC/ETH only

prev_date = (datetime.fromisoformat(date) - timedelta(days=1)).strftime("%Y-%m-%d")
premium = pd.concat([
    get_premium_index_klines(symbol, prev_date),   # warms up the 8h window
    get_premium_index_klines(symbol, date),
]).sort_index()
print(premium)

# # Linear time weight: most recent bar dominates. Size the window to 8h of bars.
step = premium.index.to_series().diff().median()
n = int(pd.Timedelta("8h") / step)
w = np.arange(1, n + 1)

p_avg = premium["close"].rolling(n).apply(lambda x: x @ w / w.sum(), raw=True)

funding_rate = p_avg + (I - p_avg).clip(-0.0005, 0.0005)
funding = pd.DataFrame({"funding_rate": funding_rate})
funding["funding_rate_annualized"] = funding["funding_rate"] * 3 * 365
funding = funding.resample("5min").last().loc[date]

                         open      high       low     close  volume  \
open_time                                                             
2026-06-09 00:00:00 -0.000622 -0.000287 -0.000797 -0.000314       0   
2026-06-09 00:01:00 -0.000549 -0.000366 -0.000653 -0.000567       0   
2026-06-09 00:02:00 -0.000251 -0.000149 -0.000624 -0.000317       0   
2026-06-09 00:03:00 -0.000442 -0.000442 -0.000805 -0.000659       0   
2026-06-09 00:04:00 -0.000550 -0.000486 -0.000836 -0.000486       0   
...                       ...       ...       ...       ...     ...   
2026-06-10 23:55:00 -0.000326 -0.000326 -0.000516 -0.000414       0   
2026-06-10 23:56:00 -0.000412 -0.000341 -0.000532 -0.000496       0   
2026-06-10 23:57:00 -0.000494 -0.000197 -0.000559 -0.000538       0   
2026-06-10 23:58:00 -0.000473 -0.000314 -0.000487 -0.000369       0   
2026-06-10 23:59:00 -0.000430 -0.000219 -0.000496 -0.000496       0   

                        close_time  quote_volume  count  taker_buy_volume  \

In [2]:
combined = pd.concat([fut_trades['fut_cumulative_volume_delta'], spot['spot_cumulative_volume_delta'], mark_5m, oi['sum_open_interest'], funding['funding_rate_annualized']], axis=1)

In [3]:
from orderflow import build_order_flow_chart

my_layout = [
    {"type": "candlestick", "title": "Price Action", "name": "Candlestick", "y_title": "Price (USDT)"},
    {"type": "line", "title": "Open Interest", "column": "sum_open_interest", "name": "OI Coins", "color": "purple", "y_title": "OI"},
    {"type": "delta_bars", "title": "Funding Rate", "column": "funding_rate_annualized", "name": "Funding", "color": "orange", "y_title": "Rate"},
    {"type": "delta_bars", "title": "Spot CVD", "column": "spot_cumulative_volume_delta", "name": "Spot Delta", "y_title": "Delta"},
    {"type": "delta_bars", "title": "Futures CVD", "column": "fut_cumulative_volume_delta", "name": "Futures Delta", "y_title": "Delta"}
]

fig = build_order_flow_chart(combined, "2026-06-14", config=my_layout)
fig.show()